# UTS Praktikum Penelusuran Informasi
## Multi-Dataset Information Retrieval System (CLI-Based)

**Tujuan:** membangun sistem pencarian dokumen dari lima sumber teks menggunakan preprocessing, representasi Bag-of-Words, indeks Whoosh, dan pemeringkatan cosine similarity.

**Sumber data:** ETD-USK, ETD-UGM, Kompas, Tempo, dan Mojok.


## 1. Instalasi dan import

In [ ]:
!pip -q install Whoosh scikit-learn requests beautifulsoup4

import os
import re
import json
import math
import shutil
import tarfile
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from collections import Counter

from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer
from whoosh.qparser import QueryParser
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Fungsi preprocessing

In [ ]:
STOPWORDS = {
    "yang", "di", "ke", "dari", "pada", "dalam", "untuk", "dan",
    "atau", "dengan", "ini", "itu", "adalah", "sebagai", "oleh"
}

def preprocess_text(text):
    # Case folding, pembersihan tanda baca, tokenisasi, dan stopword removal
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in STOPWORDS]
    return " ".join(tokens)


## 3. Siapkan folder dataset

Jalankan sel ini untuk memastikan lima folder tersedia. Kompas dan Tempo diisi dari arsip berita yang sebelumnya digunakan. Jika arsip belum ada di runtime Colab, unduh dan ekstrak terlebih dahulu.


In [ ]:
BASE_DIR = "/content/dataset"
SOURCES = ["etd-usk", "etd-ugm", "kompas", "tempo", "mojok"]

for source in SOURCES:
    os.makedirs(os.path.join(BASE_DIR, source), exist_ok=True)

print("Folder dataset siap:")
for source in SOURCES:
    folder = os.path.join(BASE_DIR, source)
    count = sum(
        1 for name in os.listdir(folder)
        if name.endswith(".txt") and os.path.isfile(os.path.join(folder, name))
    )
    print(f"{source.upper()}: {count} file .txt")


## 4. Ekstrak 100 artikel Kompas dan 100 Tempo dari arsip berita

Jalankan hanya jika arsip `newspapers-json.tgz` belum diekstrak / folder `/content/newspapers/json/2018` belum tersedia. Sel ini tidak menghapus artikel yang sudah ada; nama file baru dibuat berurutan.


In [ ]:
ARCHIVE_PATH = "/content/newspapers-json.tgz"
JSON_ROOT = "/content/newspapers/json/2018"

if not os.path.isdir(JSON_ROOT):
    archive_url = "https://huggingface.co/datasets/indonesian-nlp/id_newspapers_2018/resolve/main/newspapers-json.tgz"
    os.makedirs("/content/newspapers", exist_ok=True)
    response = requests.get(archive_url, timeout=120)
    response.raise_for_status()
    with open(ARCHIVE_PATH, "wb") as f:
        f.write(response.content)
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall("/content/newspapers")

def extract_news_articles(limit_per_source=100):
    counts = {}
    for source in ["kompas", "tempo"]:
        folder = os.path.join(BASE_DIR, source)
        files = [
            name for name in os.listdir(folder)
            if name.endswith(".txt") and os.path.isfile(os.path.join(folder, name))
        ]
        counts[source] = len(files)

    for root, _, files in os.walk(JSON_ROOT):
        for name in files:
            if not name.endswith(".json") or name.startswith("._"):
                continue
            if all(counts[s] >= limit_per_source for s in counts):
                break

            lower_name = name.lower()
            source = "kompas" if ".kompas." in lower_name else (
                "tempo" if ".tempo." in lower_name else None
            )
            if source is None or counts[source] >= limit_per_source:
                continue

            path = os.path.join(root, name)
            try:
                with open(path, "r", encoding="utf-8") as f:
                    item = json.load(f)
                if not isinstance(item, dict):
                    continue
                title = str(item.get("title", "")).strip()
                content = str(item.get("content", "")).strip()
                url = str(item.get("url", "")).strip()
                if not title or not content:
                    continue

                counts[source] += 1
                output_path = os.path.join(
                    BASE_DIR, source, f"{source}_{counts[source]:03d}.txt"
                )
                # Simpan hanya jika nama baru belum ada
                if os.path.exists(output_path):
                    continue
                with open(output_path, "w", encoding="utf-8") as f:
                    f.write(f"Judul: {title}\nSumber: {url}\n\n{content}")
            except (OSError, json.JSONDecodeError, UnicodeDecodeError):
                continue
        if all(counts[s] >= limit_per_source for s in counts):
            break

    print("Jumlah artikel saat ini:")
    for source in ["kompas", "tempo"]:
        print(f"{source.upper()}: {counts[source]}")

extract_news_articles(100)


## 5. Ambil artikel Mojok yang tersedia secara publik

Sel ini mencoba mengambil hingga 10 halaman dari situs Mojok. Struktur situs bisa berubah; periksa output dan isi file yang dihasilkan. Hormati ketentuan situs dan gunakan konten untuk tugas sesuai aturan yang berlaku.


In [ ]:
def collect_mojok_articles(max_articles=10):
    folder = os.path.join(BASE_DIR, "mojok")
    os.makedirs(folder, exist_ok=True)
    existing = {
        name for name in os.listdir(folder)
        if name.endswith(".txt") and os.path.isfile(os.path.join(folder, name))
    }
    saved = 0
    headers = {"User-Agent": "Mozilla/5.0"}
    listing_url = "https://mojok.co/konter/"

    try:
        response = requests.get(listing_url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
    except requests.RequestException as e:
        print("Tidak dapat membuka daftar Mojok:", e)
        return

    links = []
    for a in soup.select("a[href]"):
        link = urljoin(listing_url, a["href"])
        if "mojok.co/" in link and link not in links and a.get_text(strip=True):
            links.append(link)

    for link in links:
        if saved >= max_articles:
            break
        try:
            r = requests.get(link, headers=headers, timeout=20)
            if r.status_code != 200:
                continue
            page = BeautifulSoup(r.text, "html.parser")
            title_tag = page.find("h1")
            article = page.select_one("article") or page.find("main")
            if not title_tag or not article:
                continue
            title = title_tag.get_text(" ", strip=True)
            content = article.get_text(" ", strip=True)
            if len(content) < 300:
                continue
            safe_title = re.sub(r"[^a-zA-Z0-9_-]", "_", title)[:70].strip("_")
            filename = f"mojok_{safe_title or 'artikel'}.txt"
            if filename in existing:
                continue
            with open(os.path.join(folder, filename), "w", encoding="utf-8") as f:
                f.write(f"Judul: {title}\nSumber: {link}\n\n{content}")
            existing.add(filename)
            saved += 1
            print("Tersimpan:", title)
        except (requests.RequestException, OSError):
            continue
    print(f"Artikel Mojok baru disimpan: {saved}")

# Jalankan bila folder Mojok masih perlu diisi; file yang sudah ada tidak ditimpa.
collect_mojok_articles(10)


## 6. Periksa dokumen ETD-USK dan ETD-UGM


In [ ]:
# Periksa file ETD yang tersedia sebelum indexing.
for source in ["etd-usk", "etd-ugm"]:
    folder = os.path.join(BASE_DIR, source)
    print(f"\n{source.upper()}:")
    for name in sorted(os.listdir(folder)):
        if name.endswith(".txt"):
            print("-", name)


## 7. Periksa jumlah dokumen dan file kosong

In [ ]:
def report_datasets():
    total = 0
    for source in SOURCES:
        folder = os.path.join(BASE_DIR, source)
        files = [
            name for name in os.listdir(folder)
            if name.endswith(".txt") and os.path.isfile(os.path.join(folder, name))
        ]
        valid = 0
        empty = 0
        for name in files:
            path = os.path.join(folder, name)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read().strip()
            if text:
                valid += 1
            else:
                empty += 1
        total += valid
        print(f"{source.upper()}: {valid} dokumen berisi teks, {empty} file kosong")
    print("TOTAL dokumen berisi teks:", total)

report_datasets()


## 8. Buat ulang indeks Whoosh dengan preprocessing

In [ ]:
INDEX_DIR = "/content/indeks_whoosh"

def build_whoosh_index():
    if os.path.exists(INDEX_DIR):
        shutil.rmtree(INDEX_DIR)
    os.makedirs(INDEX_DIR, exist_ok=True)

    schema = Schema(
        judul=TEXT(stored=True),
        konten=TEXT(stored=True),
        sumber=ID(stored=True),
        path=ID(stored=True, unique=True)
    )
    ix = index.create_in(INDEX_DIR, schema)
    writer = ix.writer()
    total = 0

    for source in SOURCES:
        folder = os.path.join(BASE_DIR, source)
        count = 0
        if not os.path.isdir(folder):
            print(f"{source.upper()}: folder tidak ditemukan")
            continue

        for name in sorted(os.listdir(folder)):
            if not name.endswith(".txt"):
                continue
            path = os.path.join(folder, name)
            if not os.path.isfile(path):
                continue
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    raw = f.read().strip()
                if not raw:
                    continue
                processed = preprocess_text(raw)
                if not processed:
                    continue
                first_line = next((line.strip() for line in raw.splitlines() if line.strip()), name)
                writer.add_document(
                    judul=first_line[:250],
                    konten=processed,
                    sumber=source.upper(),
                    path=path
                )
                count += 1
                total += 1
            except OSError as e:
                print("File dilewati:", path, e)
        print(f"{source.upper()}: {count} dokumen diindeks")

    writer.commit()
    print(f"\nSelesai. Total dokumen diindeks: {total}")
    return total

build_whoosh_index()


## 9. Pencarian dan ranking cosine similarity (Bag-of-Words)

In [ ]:
def load_vectors_from_index():
    ix = index.open_dir(INDEX_DIR)
    docs = []
    with ix.searcher() as searcher:
        for stored in searcher.all_stored_fields():
            docs.append({
                "judul": stored["judul"],
                "konten": stored["konten"],
                "sumber": stored["sumber"],
                "path": stored["path"]
            })
    if not docs:
        raise ValueError("Indeks kosong. Periksa dataset lalu jalankan ulang indexing.")
    vectorizer = CountVectorizer()
    matrix = vectorizer.fit_transform([d["konten"] for d in docs])
    return docs, vectorizer, matrix

dokumen, vectorizer, matriks_dokumen = load_vectors_from_index()

def cari_dokumen(query, top_k=5):
    query = query.strip()
    if not query:
        print("Query tidak boleh kosong.")
        return

    query_clean = preprocess_text(query)
    if not query_clean:
        print("Query tidak memiliki kata setelah preprocessing.")
        return

    query_vector = vectorizer.transform([query_clean])
    scores = cosine_similarity(query_vector, matriks_dokumen).ravel()
    ranking = scores.argsort()[::-1]
    shown = 0

    print(f"\nHasil pencarian: {query}")
    print("=" * 65)
    for idx in ranking:
        if scores[idx] <= 0:
            continue
        doc = dokumen[idx]
        shown += 1
        print(f"\nPeringkat : {shown}")
        print(f"Judul     : {doc['judul']}")
        print(f"Sumber    : {doc['sumber']}")
        print(f"Cosine    : {scores[idx]:.4f}")
        print(f"File      : {doc['path']}")
        if shown >= top_k:
            break
    if shown == 0:
        print("Tidak ditemukan dokumen yang cocok.")


## 10. Menu CLI

In [ ]:
def main_menu():
    while True:
        print("\n=== INFORMATION RETRIEVAL SYSTEM ===")
        print("[1] Lihat jumlah dokumen")
        print("[2] Search query")
        print("[3] Keluar")
        pilihan = input("Pilih menu (1/2/3): ").strip()

        if pilihan == "1":
            report_datasets()
        elif pilihan == "2":
            query = input("Masukkan query pencarian: ")
            cari_dokumen(query, top_k=5)
        elif pilihan == "3":
            print("Program selesai.")
            break
        else:
            print("Pilihan tidak valid.")

# Jalankan menu saat ingin mencoba CLI:
# main_menu()
